In [4]:
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch, RunnablePassthrough

llm = ChatOllama(model = 'llama3.2:3b', temperature=0)


In [5]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    response_flag : Literal['positive', 'negative']

llm_struct_output = llm.with_structured_output(llm_schema)


In [6]:
result = llm_struct_output.invoke('điện thoại dỏm quá')
result.model_dump()['response_flag']


'negative'

## Chain with Conditional


In [7]:
# TASK 1 - prompt
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'Bạn là 1 chuyên gia thẩm định'),
    ('human', 'Hãy phân loại sản phẩm này là negative hay positive: {text}')
])


In [8]:
# TASK 2 - LLM
llm_struct_output = llm.with_structured_output(llm_schema)


In [9]:
# TASK 3 - [custom runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input : llm_schema) -> str:
    return input.model_dump()['response_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)


## Condition chain 1


In [10]:
# TASK 1 - PROMPT
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một biên tập viên chuyên nghiệp."),
    ("human", "Hãy tóm tắt lại nội dung sau trong 1 câu ngắn gọn: {text}")
])

# TASK 2 - LLM
llm = ChatOllama(model = 'llama3.2:3b', temperature=0)

# TASK 3 - OUTPUT PARSER
str_parser = StrOutputParser()

chain_summary = prompt_summary | llm | str_parser


## Condition chain 2


In [11]:
# TASK 1 - PROMPT
prompt_sentiment = ChatPromptTemplate.from_messages([
    ("system", "Bạn là chuyên gia phân tích phản hồi khách hàng."),
    ("human", "Hãy đánh giá sắc thái của đoạn văn sau kèm lý do ngắn: {text}")
])

# TASK 2 - LLM
llm = ChatOllama(model = 'llama3.2:3b', temperature=0)

# TASK 3 - OUTPUT PARSER
str_parser = StrOutputParser()

chain_sentiment = prompt_sentiment | llm | str_parser


## FINAL CHAIN


In [12]:
# 1. Chuỗi phân loại cảm xúc
classifier_chain = prompt_template | llm_struct_output | pydantic_json_runnable

# 2. Giữ nguyên text ban đầu và thêm nhãn phân loại vào key 'sentiment'
classify_chain = RunnablePassthrough.assign(sentiment=classifier_chain)

# 3. Điều kiện rẽ nhánh bằng RunnableBranch
conditional_chain = RunnableBranch(
    (lambda x: x['sentiment'] == 'positive', chain_summary),
    chain_sentiment
)

# 4. Chuỗi hoàn chỉnh
final_chain = classify_chain | conditional_chain


In [13]:
# Ví dụ 1: Phản hồi tích cực
input_positive = {
    "text": "Tôi rất thích chiếc điện thoại này, thiết kế đẹp, màn hình sắc nét và pin rất trâu."
}

classified_pos = classify_chain.invoke(input_positive)
label_pos = classified_pos["sentiment"]
output_pos = final_chain.invoke(input_positive)

print("--- VI DU 1: PHAN HOI TICH CUC ---")
print("Input:", input_positive["text"])
print("Nhan phan loai:", label_pos)
print("Nhanh duoc chon: chain_summary")
print("Output:", output_pos)

print("\n" + "-" * 50 + "\n")

# Ví dụ 2: Phản hồi tiêu cực
input_negative = {
    "text": "Điện thoại quá tệ, dùng một lúc đã nóng ran, pin tụt nhanh và giá quá đắt."
}

classified_neg = classify_chain.invoke(input_negative)
label_neg = classified_neg["sentiment"]
output_neg = final_chain.invoke(input_negative)

print("--- VI DU 2: PHAN HOI TIEU CUC ---")
print("Input:", input_negative["text"])
print("Nhan phan loai:", label_neg)
print("Nhanh duoc chon: chain_sentiment")
print("Output:", output_neg)


--- VI DU 1: PHAN HOI TICH CUC ---
Input: Tôi rất thích chiếc điện thoại này, thiết kế đẹp, màn hình sắc nét và pin rất trâu.
Nhan phan loai: positive
Nhanh duoc chon: chain_summary
Output: Bạn thích chiếc điện thoại này vì thiết kế đẹp, màn hình sắc nét và pin có thời lượng dài.

--------------------------------------------------

--- VI DU 2: PHAN HOI TIEU CUC ---
Input: Điện thoại quá tệ, dùng một lúc đã nóng ran, pin tụt nhanh và giá quá đắt.
Nhan phan loai: negative
Nhanh duoc chon: chain_sentiment
Output: Sắc thái của đoạn văn này là tiêu cực, có tính phán xét và chỉ trích.

Lý do: Đoạn văn sử dụng từ ngữ tiêu cực như "tế", "nóng ran", "tụt nhanh" để mô tả điện thoại, đồng thời cũng chỉ trích giá của sản phẩm. Điều này tạo ra một sắc thái tiêu cực và có thể gây ảnh hưởng tiêu cực đến hình ảnh của thương hiệu.
